# 11 · Department-Level Analysis for Manager Job Role

**Project:** Enterprise HR AI  
**Purpose:** Provide meaningful statistical analysis for the 102 employees in the `Manager` JobRole.  
In Step 10, the `Manager` role was flagged with `very_low` mapping confidence because O*NET contains 52+ disparate managerial codes, making any direct occupational mapping arbitrary and unreliable.  

> **Methodological Principle:** Rather than fabricating occupational skill profiles via fuzzy matching or generic taxonomy workarounds, Department (`Research & Development`, `Sales`, `Human Resources`) serves as the deliberate, empirical lens to analyze manager demographics, compensation, retention, and flight risks.

---

In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = os.path.join('..', 'data', 'processed', 'employee_attrition_processed.csv')
df = pd.read_csv(DATA_PATH)

print(f'Loaded dataset shape: {df.shape}')
overall_attrition_rate = (df['Attrition'] == 'Yes').mean() * 100
print(f'Company-wide benchmark attrition rate: {overall_attrition_rate:.2f}% (Step 1 baseline: 16.12%)')

Loaded dataset shape: (1470, 35)
Company-wide benchmark attrition rate: 16.12% (Step 1 baseline: 16.12%)


---
## Step 1 · Filter to Manager Role & Confirm Count

In [2]:
managers = df[df['JobRole'] == 'Manager'].copy()
mgr_count = len(managers)
print(f'Total Manager employees found: {mgr_count}')
assert mgr_count == 102, f'Expected exactly 102 managers, found {mgr_count}!'
print('CONFIRMED: Exactly 102 Manager records isolated.')

Total Manager employees found: 102
CONFIRMED: Exactly 102 Manager records isolated.


---
## Step 2 · Department Breakdown

Analyzing the distribution of the 102 managers across organizational departments.

In [3]:
dept_counts = managers['Department'].value_counts()
dept_pcts = managers['Department'].value_counts(normalize=True) * 100

dept_summary = pd.DataFrame({
    'Headcount': dept_counts,
    'Percentage (%)': dept_pcts.round(2)
})
dept_summary.index.name = 'Department'

print('=== MANAGER HEADCOUNT BY DEPARTMENT ===')
print(dept_summary.to_string())
print(f'\nTotal Managers: {dept_summary["Headcount"].sum()}')

=== MANAGER HEADCOUNT BY DEPARTMENT ===
                        Headcount  Percentage (%)
Department                                       
Research & Development         54           52.94
Sales                          37           36.27
Human Resources                11           10.78

Total Managers: 102


---
## Step 3 · Metrics by Department: Attrition, Income, Satisfaction, OverTime

Evaluating key risk and retention factors across departments and comparing against company-wide baselines.

In [4]:
# Overall manager metrics
total_mgr_att_count = (managers['Attrition'] == 'Yes').sum()
total_mgr_att_rate = (managers['Attrition'] == 'Yes').mean() * 100
total_mgr_ot_rate = (managers['OverTime'] == 'Yes').mean() * 100

print('=== MANAGER OVERALL VS COMPANY BENCHMARK ===')
print(f'Company Baseline Attrition Rate : {overall_attrition_rate:.2f}%')
print(f'Manager Overall Attrition Rate  : {total_mgr_att_rate:.2f}% ({total_mgr_att_count} / {mgr_count} leavers)')
print(f'Manager Overall OverTime Rate   : {total_mgr_ot_rate:.2f}% ({(managers["OverTime"] == "Yes").sum()} / {mgr_count} working OT)')
print('FLAG: Manager attrition (4.90%) is substantially LOWER than company-wide average (16.12%).')
print()

# Department-level aggregation
dept_groups = managers.groupby('Department')

metrics_list = []
for dept, group in dept_groups:
    n = len(group)
    att_count = (group['Attrition'] == 'Yes').sum()
    att_rate = (group['Attrition'] == 'Yes').mean() * 100
    mean_income = group['MonthlyIncome'].mean()
    mean_job_sat = group['JobSatisfaction'].mean()
    mean_wlb = group['WorkLifeBalance'].mean()
    mean_tenure = group['YearsAtCompany'].mean()
    ot_count = (group['OverTime'] == 'Yes').sum()
    ot_rate = (group['OverTime'] == 'Yes').mean() * 100
    
    metrics_list.append({
        'Department': dept,
        'Headcount': n,
        'Attrition Count': att_count,
        'Attrition Rate (%)': round(att_rate, 2),
        'Mean Monthly Income ($)': round(mean_income, 2),
        'Mean Job Satisfaction (1-4)': round(mean_job_sat, 2),
        'Mean Work-Life Balance (1-4)': round(mean_wlb, 2),
        'Mean Years at Company': round(mean_tenure, 2),
        'OverTime Rate (%)': round(ot_rate, 2)
    })

# Add Overall Manager row for comparison
metrics_list.append({
    'Department': 'All Managers (Total)',
    'Headcount': mgr_count,
    'Attrition Count': total_mgr_att_count,
    'Attrition Rate (%)': round(total_mgr_att_rate, 2),
    'Mean Monthly Income ($)': round(managers['MonthlyIncome'].mean(), 2),
    'Mean Job Satisfaction (1-4)': round(managers['JobSatisfaction'].mean(), 2),
    'Mean Work-Life Balance (1-4)': round(managers['WorkLifeBalance'].mean(), 2),
    'Mean Years at Company': round(managers['YearsAtCompany'].mean(), 2),
    'OverTime Rate (%)': round(total_mgr_ot_rate, 2)
})

metrics_df = pd.DataFrame(metrics_list)
print('=== MANAGER METRICS BY DEPARTMENT ===')
print(metrics_df.to_string(index=False))

# Cross-tabulate OverTime vs Attrition within Managers
print('\n=== OVERTIME VS ATTRITION WITHIN MANAGERS ===')
ot_ct = pd.crosstab(managers['OverTime'], managers['Attrition'], margins=True)
print(ot_ct)
ot_att_rate = (managers[managers['OverTime'] == 'Yes']['Attrition'] == 'Yes').mean() * 100
no_ot_att_rate = (managers[managers['OverTime'] == 'No']['Attrition'] == 'Yes').mean() * 100
print(f'\nAttrition rate among Managers working OverTime    : {ot_att_rate:.2f}% (4 / 27)')
print(f'Attrition rate among Managers NOT working OverTime: {no_ot_att_rate:.2f}% (1 / 75)')
print('KEY INSIGHT: 4 out of 5 (80%) Manager leavers worked OverTime! The Step 8 SHAP #1 driver holds firmly.')

=== MANAGER OVERALL VS COMPANY BENCHMARK ===
Company Baseline Attrition Rate : 16.12%
Manager Overall Attrition Rate  : 4.90% (5 / 102 leavers)
Manager Overall OverTime Rate   : 26.47% (27 / 102 working OT)
FLAG: Manager attrition (4.90%) is substantially LOWER than company-wide average (16.12%).

=== MANAGER METRICS BY DEPARTMENT ===
            Department  Headcount  Attrition Count  Attrition Rate (%)  Mean Monthly Income ($)  Mean Job Satisfaction (1-4)  Mean Work-Life Balance (1-4)  Mean Years at Company  OverTime Rate (%)
       Human Resources         11                0                0.00                 18088.64                         2.82                          2.91                  16.27              36.36
Research & Development         54                3                5.56                 17130.33                         2.65                          2.76                  13.52              24.07
                 Sales         37                2                5.41  

---
## Step 4 · Plain-Language HR Summary

**Executive Insights for HR Leadership & People Analytics:**

Managers represent the most stable, highly retained talent pool in the organization, with an overall attrition rate of just **4.90%** (5 leavers across 102 leaders) compared to the company-wide baseline of **16.12%**. Across departments, Human Resources experienced zero manager turnover (0.00% across 11 leaders), while Research & Development (5.56%, 3 of 54) and Sales (5.41%, 2 of 37) showed very low, nearly identical attrition levels. This superior retention is anchored by substantial compensation (averaging over $17,000/month across all departments) and exceptional organizational tenure (averaging 14.4 years at the company, peaking at 16.3 years in HR), which strongly aligns with our Step 8 SHAP findings where senior Job Level, high tenure, and higher income heavily dampen flight risk.

However, the Step 8 SHAP finding identifying **OverTime** as the organization's #1 attrition driver holds with remarkable precision even in this executive cohort: **4 out of the 5 managers who left (80%) worked overtime**, resulting in an attrition rate of **14.81%** among overtime-burdened managers versus a negligible **1.33%** (1 of 75) for those who did not. While overall manager satisfaction (2.71/4) and work-life balance (2.77/4) are moderate across R&D, Sales, and HR, sustained overtime represents the primary catalyst that destabilizes even well-compensated, loyal leadership talent. HR should focus manager retention efforts not on general compensation increases, but specifically on leadership workload audits and executive burnout prevention for the 26.5% of managers logging regular overtime.

In [5]:
print('=== PLAIN-LANGUAGE SUMMARY (HR LEADERSHIP BRIEFING) ===\n')
summary_text = (
    'Managers represent the most stable, highly retained talent pool in the organization, '
    'with an overall attrition rate of just 4.90% (5 leavers across 102 leaders) compared to '
    'the company-wide baseline of 16.12%. Across departments, Human Resources experienced zero '
    'manager turnover (0.00% across 11 leaders), while Research & Development (5.56%, 3 of 54) '
    'and Sales (5.41%, 2 of 37) showed very low, nearly identical attrition levels. This superior '
    'retention is anchored by substantial compensation (averaging over $17,000/month across all '
    'departments) and exceptional organizational tenure (averaging 14.4 years at the company, '
    'peaking at 16.3 years in HR), which strongly aligns with our Step 8 SHAP findings where '
    'senior Job Level, high tenure, and higher income heavily dampen flight risk.\n\n'
    'However, the Step 8 SHAP finding identifying OverTime as the organization\'s #1 attrition '
    'driver holds with remarkable precision even in this executive cohort: 4 out of the 5 managers '
    'who left (80%) worked overtime, resulting in an attrition rate of 14.81% among overtime-burdened '
    'managers versus a negligible 1.33% (1 of 75) for those who did not. While overall manager '
    'satisfaction (2.71/4) and work-life balance (2.77/4) are moderate across R&D, Sales, and HR, '
    'sustained overtime represents the primary catalyst that destabilizes even well-compensated, '
    'loyal leadership talent. HR should focus manager retention efforts not on general compensation '
    'increases, but specifically on leadership workload audits and executive burnout prevention '
    'for the 26.5% of managers logging regular overtime.'
)
print(summary_text)

=== PLAIN-LANGUAGE SUMMARY (HR LEADERSHIP BRIEFING) ===

Managers represent the most stable, highly retained talent pool in the organization, with an overall attrition rate of just 4.90% (5 leavers across 102 leaders) compared to the company-wide baseline of 16.12%. Across departments, Human Resources experienced zero manager turnover (0.00% across 11 leaders), while Research & Development (5.56%, 3 of 54) and Sales (5.41%, 2 of 37) showed very low, nearly identical attrition levels. This superior retention is anchored by substantial compensation (averaging over $17,000/month across all departments) and exceptional organizational tenure (averaging 14.4 years at the company, peaking at 16.3 years in HR), which strongly aligns with our Step 8 SHAP findings where senior Job Level, high tenure, and higher income heavily dampen flight risk.

However, the Step 8 SHAP finding identifying OverTime as the organization's #1 attrition driver holds with remarkable precision even in this executive 